In [1]:
import pandas as pd
import glob

# Path to AMRFinder result files
files = glob.glob("amrfinder_results/*.tsv")

dfs = []

for f in files:
    df = pd.read_csv(f, sep="\t")
    
    # Extract genome ID from filename
    genome = f.split("/")[-1].replace("_amrfinder.tsv", "")
    df["Genome_ID"] = genome
    
    dfs.append(df)

amr_all = pd.concat(dfs, ignore_index=True)

print("Total AMR hits:", len(amr_all))
print("Total genomes with AMR:", amr_all["Genome_ID"].nunique())


Total AMR hits: 6883
Total genomes with AMR: 811


In [2]:
amr_counts = (
    amr_all
    .groupby("Genome_ID")
    .size()
    .reset_index(name="Total_AMR_genes")
)


In [4]:
amr_all.columns


Index(['Protein id', 'Element symbol', 'Element name', 'Scope', 'Type',
       'Subtype', 'Class', 'Subclass', 'Method', 'Target length',
       'Reference sequence length', '% Coverage of reference',
       '% Identity to reference', 'Alignment length',
       'Closest reference accession', 'Closest reference name',
       'HMM accession', 'HMM description', 'Genome_ID'],
      dtype='str')

In [5]:
amr_counts = (
    amr_all
    .groupby("Genome_ID")
    .size()
    .reset_index(name="Total_AMR_genes")
)


In [6]:
class_counts = (
    amr_all
    .groupby("Genome_ID")["Class"]
    .nunique()
    .reset_index(name="Unique_AMR_classes")
)


In [7]:
class_counts["MDR_status"] = class_counts["Unique_AMR_classes"] >= 3
class_counts["MDR_status"] = class_counts["MDR_status"].map({True: "Yes", False: "No"})


In [8]:
metadata = amr_counts.merge(class_counts, on="Genome_ID", how="outer")


In [10]:
import os

genomes = [f.replace(".fasta", "") for f in os.listdir("fna") if f.endswith(".fasta")]

all_genomes = pd.DataFrame({"Genome_ID": genomes})

metadata = all_genomes.merge(metadata, on="Genome_ID", how="left")

metadata.fillna({
    "Total_AMR_genes": 0,
    "Unique_AMR_classes": 0,
    "MDR_status": "No"
}, inplace=True)


,Genome_ID,Total_AMR_genes,Unique_AMR_classes,MDR_status
0,GCA_015890145.1_PDT000902810.1_genomic,5.0,3.0,Yes
1,GCA_023596865.1_PDT001316542.1_genomic,5.0,3.0,Yes
2,GCA_024024495.1_PDT001336459.1_genomic,15.0,6.0,Yes
3,GCF_048823995.1_ASM4882399v1_genomic,3.0,3.0,Yes
4,GCF_048823355.1_ASM4882335v1_genomic,3.0,3.0,Yes
...,...,...,...,...
834,GCF_048822995.1_ASM4882299v1_genomic,3.0,3.0,Yes
835,GCA_047204735.1_ASM4720473v1_genomic,7.0,6.0,Yes
836,GCA_023596305.1_PDT001316535.1_genomic,5.0,3.0,Yes
837,GCA_047202635.1_ASM4720263v1_genomic,7.0,4.0,Yes


In [11]:
metadata.to_csv("tree_metadata.tsv", sep="\t", index=False)



In [22]:
with open("itol_MDR.txt", "w") as f:
    f.write("DATASET_BINARY\n")
    f.write("SEPARATOR TAB\n")
    f.write("DATASET_LABEL\tMDR_status\n")
    f.write("COLOR\t#8B0000\n")
    f.write("FIELD_LABELS\tMDR\n")
    f.write("FIELD_COLORS\t#8B0000\n")
    f.write("FIELD_SHAPES\t1\n")
    f.write("DATA\n")

    for _, row in metadata.iterrows():
        val = 1 if row["MDR_status"] == "Yes" else -1
        f.write(f"{row['Genome_ID']}\t{val}\n")



In [15]:
with open("itol_AMR_heatmap.txt", "w") as f:
    f.write("DATASET_HEATMAP\n")
    f.write("SEPARATOR TAB\n")
    f.write("DATASET_LABEL\tTotal_AMR_genes\n")
    f.write("COLOR\t#0000ff\n")
    f.write("FIELD_LABELS\tAMR_Count\n")
    f.write("DATA\n")
    
    for _, row in metadata.iterrows():
        f.write(f"{row['Genome_ID']}\t{row['Total_AMR_genes']}\n")


In [16]:
metadata["Total_AMR_genes"].describe()

count    839.000000
mean       8.203814
std        5.740422
min        0.000000
25%        3.000000
50%        7.000000
75%       13.000000
max       25.000000
Name: Total_AMR_genes, dtype: float64

In [1]:
import pandas as pd
import numpy as np
from Bio import Phylo
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import kruskal


In [2]:
metadata = pd.read_csv("tree_metadata.tsv", sep="\t")
metadata.head()


,Genome_ID,Total_AMR_genes,Unique_AMR_classes,MDR_status
0,GCA_015890145.1_PDT000902810.1_genomic,5.0,3.0,Yes
1,GCA_023596865.1_PDT001316542.1_genomic,5.0,3.0,Yes
2,GCA_024024495.1_PDT001336459.1_genomic,15.0,6.0,Yes
3,GCF_048823995.1_ASM4882399v1_genomic,3.0,3.0,Yes
4,GCF_048823355.1_ASM4882335v1_genomic,3.0,3.0,Yes


In [4]:
tree = Phylo.read("core_tree.newick", "newick")

tips = [term.name for term in tree.get_terminals()]
len(tips)


839

In [5]:
n = len(tips)
dist_matrix = np.zeros((n, n))

for i, t1 in enumerate(tips):
    for j, t2 in enumerate(tips):
        dist_matrix[i, j] = tree.distance(t1, t2)

condensed = squareform(dist_matrix)


In [6]:
Z = linkage(condensed, method="average")

clusters = fcluster(Z, t=4, criterion='maxclust')

cluster_df = pd.DataFrame({
    "Genome_ID": tips,
    "Cluster": clusters
})


In [7]:
test_df = metadata.merge(cluster_df, on="Genome_ID")


In [8]:
groups = [
    test_df[test_df["Cluster"] == c]["Total_AMR_genes"]
    for c in test_df["Cluster"].unique()
]

stat, p = kruskal(*groups)

print("Kruskal-Wallis H =", stat)
print("p-value =", p)


Kruskal-Wallis H = 69.69234317569385
p-value = 4.967470042905756e-15


In [9]:
test_df.groupby("Cluster")["Total_AMR_genes"].describe()


,count,mean,std,min,25%,50%,75%,max
Cluster,,,,,,,,
1,3.0,7.000000,5.196152,4.0,4.00,4.0,8.5,13.0
2,318.0,9.389937,6.736980,0.0,4.00,8.0,15.0,25.0
3,30.0,1.633333,2.370557,0.0,0.25,1.0,2.0,13.0
4,488.0,7.842213,4.795371,0.0,3.00,7.0,13.0,18.0
